# 11 — Dependency Parsing
**Goal:** Understand grammatical relationships between words.

Dependency parsing goes one level deeper than POS tagging (Ch. 10): instead of only labeling *what* each word is (noun, verb, …), it figures out *how* words connect — which word modifies which, and who is the subject, the verb, and the object of a sentence.

**Why it matters for resumes / ATS:** a bullet like *"Reduced inference latency by 40%"* carries its meaning in the *relationships* between words, not the words alone. The dependency structure tells a parser that the candidate (implied subject) **reduced** *latency* (object) *by 40%* (modifier) — the exact (subject → action → result) pattern an ATS wants to rebuild. This is the first technique in the series that understands sentence **structure**, not just word labels.

## 1. Dependency Tree Basics

Every sentence can be drawn as a **tree**: one word is the **root** (usually the main verb), and every other word points to its **head** — the word it depends on. The connection between two words carries a **dependency label** (`dep_`) that names the relationship:

| Label | Meaning | Example |
|---|---|---|
| `ROOT` | the tree's root — the main verb | *developed* |
| `nsubj` | nominal subject | *scientist* → developed |
| `dobj` | direct object | *models* → developed |
| `det` | determiner (article) | *The* → scientist |
| `compound` | noun-noun compound | *data* / *ML* → scientist / models |

**What the code does:** loads spaCy's small English model, parses one sentence, and prints each token next to its **head**, its **dependency label**, and its **children** (the words that depend on it). Reading the output, `developed` is the ROOT with children `scientist` (subject) and `models` (object) — the sentence's *who-did-what* backbone, recovered purely from grammar.

In [1]:
import spacy
nlp = spacy.load("en_core_web_sm")
doc = nlp("The data scientist developed ML models")
for token in doc:
    children = [c.text for c in token.children]
    print(f"  {token.text:10s} head:{token.head.text:10s} dep:{token.dep_:10s} children:{children}")

  The        head:scientist  dep:det        children:[]
  data       head:scientist  dep:compound   children:[]
  scientist  head:developed  dep:nsubj      children:['The', 'data']
  developed  head:developed  dep:ROOT       children:['scientist', 'models']
  ML         head:models     dep:compound   children:[]
  models     head:developed  dep:dobj       children:['ML']


## 2. Subject-Verb-Object Extraction

The (subject, verb, object) triple is the **semantic skeleton** of a sentence — and it maps directly onto a resume bullet: *what was done (verb) to what (object)*.

**What the code does:** `extract_svo()` walks every token; whenever it finds a VERB, it scans the verb's *children* for:
- a **subject** (`nsubj` / `nsubjpass` — the passive label also catches forms like "was reduced")
- an **object** (`dobj`, `attr`, or `pobj`)

It returns `(subject, verb.lemma_, object)` — using the **lemma** so "developed" / "developing" / "develops" all normalize to `develop`.

**Try it:** note how `"Team reduced latency by 40%"` → `('Team', 'reduce', 'latency')` — the *by 40%* modifier is intentionally dropped because we only extract the triple. In a real ATS, the triple is what gets matched against job-description requirements (action + object), while the modifier becomes a quantified-impact bonus.

In [4]:
def extract_svo(sent):
    doc = nlp(sent)
    triples = []
    for t in doc:
        if t.pos_ == "VERB":
            subj = next((c.text for c in t.children if c.dep_ in ("nsubj","nsubjpass")), None)
            obj = next((c.text for c in t.children if c.dep_ in ("dobj","attr","pobj")), None)
            if subj: triples.append((subj, t.lemma_, obj))
    return triples

for s in ["Data scientist developed ML models", "Team reduced latency by 40%", "Engineer deployed the app"]:
    print(f"'{s}' -> {extract_svo(s)}")

'Data scientist developed ML models' -> [('scientist', 'develop', 'models')]
'Team reduced latency by 40%' -> [('Team', 'reduce', 'latency')]
'Engineer deployed the app' -> [('Engineer', 'deploy', 'app')]


## 3. Noun Chunks

A **noun chunk** is a noun plus all the words that hang off it as modifiers: *"Senior data scientist"*, *"strong Python skills"*. spaCy's `doc.noun_chunks` groups these automatically from the dependency tree — no regex needed.

**Why it matters:** job titles and skill phrases on a resume are almost always noun chunks. Extracting chunks gives clean, complete phrases ("Senior data scientist") instead of loose tokens ("senior", "data", "scientist") — far better for matching against JD keywords and for feeding skill-extraction models.

**What the code does:** parses one phrase and prints each chunk with its **root** — the central noun the chunk is built around (`scientist`, `skills`).

In [3]:
doc = nlp("Senior data scientist with strong Python skills")
for chunk in doc.noun_chunks:
    print(f"'{chunk}' -> root: {chunk.root.text}")

'Senior data scientist' -> root: scientist
'strong Python skills' -> root: skills


## Key Insight

**Dependencies reveal relationships that regex can't — *who did what to what*.**

Regex sees *patterns of characters*; dependency parsing sees *structure of meaning*. A regex could never tell you that in "Data scientist developed ML models" the *developer* is the scientist and the *thing developed* is the models. That structure is exactly what an ATS needs to rebuild a structured profile (title, skills, achievements) from free-form bullet points.

This chapter is also the bridge to what comes next: dependency output feeds phrase **chunking** (Block C), and the (subject → verb → object) triples are the natural units to turn into keyword/vector features for resume–JD matching.